In [22]:
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import StandardScaler

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [23]:
# read data

df = pd.read_csv('src/breast_cancer_diagnostic.data', skiprows=2, header=None)

X = df.iloc[:, :-1] # everything except last column
y = df.iloc[:, -1] # last col
y = y.replace({1: 0, 2: 1})

In [24]:

# using state=42 makes result reproducable. remove for random
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42
)

# scale the values
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# verify shape 18-12
print(X_train.shape)
print(X_test.shape)

(341, 31)
(228, 31)


In [25]:
y_train = y_train.values
y_test = y_test.values

X_train = np.hstack((np.ones((X_train.shape[0], 1)), X_train))
X_test = np.hstack((np.ones((X_test.shape[0], 1)), X_test))


print(y_train.shape)
print(X_train.shape)

(341,)
(341, 32)


In [26]:
# need a weight for all inputs BP Cholesterol Age Pregnant (w1,w2,w3,w4)
# need w0 (bias)
# score = w1*BP + w2*Cholesterol + w3*Age + w4*Pregnant + b

prev_loss = 0
current_loss = 0
threshold = 1e-5

fail_safe = 5000
iterations = 0
learning_rate = 0.01

weights = np.zeros(X_train.shape[1])

while iterations < fail_safe:
    # 1. predict
    z = np.dot(X_train, weights)
    y_pred = sigmoid(z)

    # 2. calculate loss
    m = len(y_train)
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    current_loss = -(1/m) * np.sum(
        y_train * np.log(y_pred) + (1 - y_train) * np.log(1 - y_pred)
    )

    # 3. stop if converged
    if iterations > 0 and abs(current_loss - prev_loss) < threshold:
        print(f"Iterations: {iterations} | current_loss: {current_loss}")
        break

    # 4. check error
    error = y_pred - y_train

    # 5. adjust weight
    gradient = np.dot(X_train.T, error) / m
    weights = weights - learning_rate * gradient

    # 6. update for next loop
    prev_loss = current_loss
    iterations += 1

    print(f"Iterations: {iterations} | current_loss: {current_loss}")


Iterations: 1 | current_loss: 0.6931471805599453
Iterations: 2 | current_loss: 0.6743012662384336
Iterations: 3 | current_loss: 0.6565922400007562
Iterations: 4 | current_loss: 0.6399440699785177
Iterations: 5 | current_loss: 0.6242825829705712
Iterations: 6 | current_loss: 0.6095365575116029
Iterations: 7 | current_loss: 0.5956384928747418
Iterations: 8 | current_loss: 0.5825250795132447
Iterations: 9 | current_loss: 0.5701374200118432
Iterations: 10 | current_loss: 0.5584210577875905
Iterations: 11 | current_loss: 0.5473258683498436
Iterations: 12 | current_loss: 0.5368058598547877
Iterations: 13 | current_loss: 0.5268189197173517
Iterations: 14 | current_loss: 0.5173265344407533
Iterations: 15 | current_loss: 0.5082935016667272
Iterations: 16 | current_loss: 0.49968764704383967
Iterations: 17 | current_loss: 0.49147955375317126
Iterations: 18 | current_loss: 0.48364230914646544
Iterations: 19 | current_loss: 0.4761512706337396
Iterations: 20 | current_loss: 0.4689838514272926
Iterat

In [27]:
# run the algo again but on test data
z_test = np.dot(X_test, weights)
y_prob = sigmoid(z_test)

# change probability into class label. Threshold is 0.5
y_pred = (y_prob >= 0.5).astype(int)

# compare accuracy
accuracy = np.mean(y_pred == y_test)

print(f"Accurary: {accuracy}")

Accurary: 0.9912280701754386


In [28]:
print("weights:")

for i, w in enumerate(weights):
    print(f"    w{i}: {w}")

weights:
    w0: 0.3201289456552075
    w1: -0.06133194618629895
    w2: -0.4317591957472337
    w3: -0.4591533692430215
    w4: -0.42643298070453345
    w5: -0.42227007121470733
    w6: -0.1610653646933405
    w7: -0.054049850348973624
    w8: -0.38980209611269145
    w9: -0.5291458246053311
    w10: -0.06640599860871703
    w11: 0.26954331686247573
    w12: -0.46325472790146544
    w13: -0.03218777309339583
    w14: -0.3739055106153073
    w15: -0.374613559641276
    w16: -0.03681468292065098
    w17: 0.18974924073751037
    w18: 0.09548149045103302
    w19: -0.09087140333874084
    w20: 0.11692282439058371
    w21: 0.2953968632324134
    w22: -0.5408059149915605
    w23: -0.6224210010099552
    w24: -0.512421220569435
    w25: -0.49545233107299996
    w26: -0.4065709269807844
    w27: -0.21628915078339592
    w28: -0.4297145349684589
    w29: -0.5332239154472072
    w30: -0.45093037695239213
    w31: -0.06678895169729404
